In [ ]:
# Cell 1 — Load cleaned data
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("../data/processed/customer_support_tickets_clean.csv")
print(f"Duplicated tickets: {df.duplicated().sum()}")
invalid_type = df["Ticket Type"].fillna("").str.strip().eq("").sum()
print(f"Number of invalid Tickets due to missing Ticket Type: {invalid_type}")
invalid_description = df["Ticket Description"].fillna("").str.strip().eq("").sum()
print(f"Number of invalid Tickets due to missing Ticket Description: {invalid_description}")

Duplicated ticekts: 0
Number of invalid Tickets due to missing Ticket Type: 0
Number of invalid Tickets due to missing Ticket Description: 0


In [2]:
# Cell 2 — Define X and Y
X = df["Ticket Description"]
y = df["Ticket Type"]

In [3]:
# Cell 3 — Create 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state= 42
)

## TF-IDF Purpose

- TF-IDF converts the training ticket descriptions from text into numerical features that a machine-learning model can use.
- The vectorizer is fitted only on the training descriptions so that it learns its vocabulary and IDF statistics without using information from the test set.
- The fitted vectorizer is then used to transform the test descriptions using the same learned feature space. The vectorizer is not fitted again on the test data, which helps prevent data leakage.

In [4]:
# Cell 4 — Create Vectorizer
vectorizer = TfidfVectorizer()

In [5]:
# Cell 5 — Fit and transform training data
X_train_tfidf = vectorizer.fit_transform(X_train)

In [6]:
# Cell 6 — Transform test data
X_test_tfidf = vectorizer.transform(X_test)

In [7]:
# Cell 7 — Inspect matrix dimensions
print(f"Number of training tickets: {X_train_tfidf.shape[0]} - Number of TF-IDF features/terms: {X_train_tfidf.shape[1]}.")
print(f"Number of test tickets: {X_test_tfidf.shape[0]} - Number of TF-IDF features/terms: {X_test_tfidf.shape[1]}.")
if X_train_tfidf.shape[1] == X_test_tfidf.shape[1]:
    print("Columns match.")
else:
    print("Columns do not match")

Number of training tickets: 6775 - Number of TF-IDF features/terms: 6290.
Number of test tickets: 1694 - Number of TF-IDF features/terms: 6290.
Columns match.


In [8]:
# Cell 8 — Inspect one transformed ticket
ticket_index = 0

print("Original ticket:")
print(X_train.iloc[ticket_index])

ticket_vector = X_train_tfidf[ticket_index]

feature_names = vectorizer.get_feature_names_out()

nonzero_indices = ticket_vector.nonzero()[1]

for index in nonzero_indices:
    print(feature_names[index], ticket_vector[0, index])

Original ticket:
I'm having an issue with the {product_purchased}. Please assist.

I'm having an issue with the {product_purchased}. Please assist.

I'm having an issue with the {product_purch I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.
having 0.23782442047591448
an 0.23368552964140812
issue 0.2627390364765316
with 0.2180607071172702
the 0.2411453059033773
product_purchased 0.17847259217292574
please 0.15009121823113122
assist 0.16228053371290224
product_purch 0.37108159965670806
ve 0.08643506155308012
noticed 0.1761838199513192
that 0.1363693002177221
occurs 0.2693649599290409
consistently 0.2708788040566095
when 0.21361151286083432
use 0.1918795557622699
specific 0.2065524730531365
feature 0.26366770399699113
or 0.15377215755762128
application 0.2344302491919427
on 0.11294451563875611
my 0.08153319824885324


In [9]:
# Cell 9 — Inspect the vocabulary/features
feature_names = vectorizer.get_feature_names_out()

print(f"Number of features: {len(feature_names)}")
print(feature_names[:20])

Number of features: 6290
['00' '000' '0000' '0002' '0010' '001250' '0014' '0020' '0025' '0033s'
 '003fc58d' '0041' '0045' '00606065' '00am' '00pm' '01' '0100' '0126'
 '0143']


## Day 24 Conclusions

- The training and testing ticket descriptions were converted from text into numerical TF-IDF features.

- The TF-IDF vectorizer was fitted only on the training descriptions. The test descriptions were transformed using the already-fitted vectorizer so that information from the test set was not used to learn the vocabulary or IDF statistics.

- The training TF-IDF matrix contains 6775 tickets and 6290 features, while the testing matrix contains 1694 tickets and the same 6290 features.

- Both matrices contain the same number of feature columns because the same fitted vectorizer was used for the training and testing data.

- Inspection of an individual transformed ticket showed which terms from its description received nonzero TF-IDF weights.

- The numerical training features are now ready to be used with the training labels to train the ticket classification model.